Project 1: Weather Data Logger

Objective: Use an API to retrieve weather data for a specific city and store the data in a text file.

Instructions:

    Get an API key from a weather service like OpenWeatherMap.
    Use the requests package to fetch current weather data (e.g., temperature, humidity, weather conditions) for a city entered by the user.
    Write the data to a text file, appending a new entry each time the script is run.
    Ensure the file contains a timestamp of when the data was retrieved.
    Read the file, parse the API response to extract specific fields (e.g., temperature, weather description).
    Print the data for the user. Ensure that each log entry is well-formatted and easy to read.


In [ ]:
# API PROJECT 1

import requests
from google.colab import userdata
apikey = userdata.get('openweathermaps')

lat = 40.736754872068886
lon = -74.17779514088078
#40.736754872068886, -74.17779514088078

url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={apikey}"
print(url)
response = requests.get(url)
data = response.json()

# we have access to data (a dictionary)
# i want you to get temp from data
ktemp = data["main"]["temp"]
kfeelslike = data["main"]["feels_like"]

ftemp = k_to_f(ktemp)
ffeelslike = k_to_f(kfeelslike)

humidity = data["main"]["humidity"]
condition = data["weather"][0]["description"]

print(ftemp, humidity, condition)

filename = "weather.txt"

with open(filename, "a") as f:
  onestring = str(ftemp) + " " + str(humidity) + " " + condition + "\n"
  f.write(onestring)

# for each in data:
#   print(each)


https://api.openweathermap.org/data/2.5/weather?lat=40.736754872068886&lon=-74.17779514088078&appid=82b633e8db7f0e1f4656270ecb529849
62.096 90 overcast clouds


Project 2: Currency Exchange Rate Tracker

Objective: Build a Python application that retrieves currency exchange rates using an API and stores the data in an SQLite database.

Instructions:

    Obtain an API key from a service like ExchangeRate-API.
    Allow the user to input two currency codes (e.g., USD, EUR).
    Use the requests package to fetch the exchange rate between the two currencies.
    Store the exchange rate, along with a timestamp and the currency codes, in an SQLite database.
    Ensure the script updates the database with new data if it is run multiple times.


In [ ]:
import requests
import sqlite3
import time

# 1. Get user input for currencies
from_cur = input("Enter from currency code (e.g. USD): ").strip().lower()
to_cur = input("Enter to currency code (e.g. EUR): ").strip().lower()

# 2. Fetch the exchange rate from the API
url = f"https://cdn.jsdelivr.net/npm/@fawazahmed0/currency-api@latest/v1/currencies/{from_cur}.json"
response = requests.get(url)
data = response.json()

# Extract the specific rate
rate = data[from_cur][to_cur]
print(f"Current rate from {from_cur.upper()} to {to_cur.upper()} is {rate}")

# 3. Store the data in SQLite database
conn = sqlite3.connect('rates.db')
cursor = conn.cursor()

# Create table if it doesn't exist
cursor.execute('''
    CREATE TABLE IF NOT EXISTS rates (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        from_currency TEXT,
        to_currency TEXT,
        rate REAL,
        timestamp TEXT
    )
''')

# Insert the current exchange rate with a timestamp
timestamp = time.strftime('%Y-%m-%d %H:%M:%S')
cursor.execute('''
    INSERT INTO rates (from_currency, to_currency, rate, timestamp)
    VALUES (?, ?, ?, ?)
''', (from_cur.upper(), to_cur.upper(), rate, timestamp))

conn.commit()
conn.close()

print("Exchange rate has been successfully saved to the database!")

Fact inserted successfully


Project 3: News Aggregator with Database Storage

Build a news aggregator that fetches headlines from the NewsAPILinks to an external site. (API key required).

Tasks:

    Write a Python program that:
        a. Prompts the user to enter a news topic (e.g., "technology", "sports").
        b. Retrieves the top 5 headlines related to the topic using the requests package.
        c. Stores the headlines, descriptions, and publication dates in an SQLite database.
        d. Reads back the data from the database and displays it to the user in a readable format.
        e. Ensures that duplicate headlines are not stored in the database.

Sample Output:

Enter news topic: technology

Top Headlines:

1. Title: Tech Innovation in 2023
   Description: A deep dive into upcoming tech trends.
   Published At: 2023-09-28

2. Title: AI Revolutionizes Industry
   Description: How AI is changing the landscape of various sectors.
   Published At: 2023-09-27

...

Data has been saved to the database.

Challenge:

    Add functionality to allow the user to:
        a. Search for headlines by keyword.
        b. Delete old headlines from the database based on a date range.


In [ ]:
import sqlite3
from requests import *
from google.colab import userdata

DB_FILE = "news_database.db"

# --- SQLite Database Functions ---
def init_db():
    """Initializes the SQLite database table if it doesn't exist."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT UNIQUE,
            description TEXT,
            published_at TEXT
        )
    ''')
    conn.commit()
    conn.close()

def save_headlines(new_articles):
    """Saves articles to SQLite while ignoring duplicate titles using INSERT OR IGNORE."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    saved_count = 0
    for article in new_articles:
        title = article.get("title")
        description = article.get("description") or "No description available."
        published_at = article.get("publishedAt")

        if title:
            if published_at:
                published_at = published_at[:10]

            try:
                cursor.execute('''
                    INSERT OR IGNORE INTO articles (title, description, published_at)
                    VALUES (?, ?, ?)
                ''', (title, description, published_at))
                # If the row was actually inserted (changes count will increase)
                if conn.total_changes > saved_count:
                    saved_count = conn.total_changes
            except sqlite3.Error as e:
                print(f"Database insertion error: {e}")

    conn.commit()
    conn.close()
    print(f"\nData has been saved to the database. (Saved {saved_count} new unique articles)")

def display_all_articles():
    """Displays all articles currently stored in the SQLite database."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT title, description, published_at FROM articles")
    rows = cursor.fetchall()
    conn.close()

    print("\nTop Headlines from Database:")
    if not rows:
        print("No articles found in the database.")
        return

    for idx, (title, description, published_at) in enumerate(rows, 1):
        print(f"{idx}. Title: {title}")
        print(f"   Description: {description}")
        print(f"   Published At: {published_at}\n")

# --- Challenge Functions ---
def search_headlines_by_keyword(keyword):
    """Searches for stored headlines containing a keyword using SQL LIKE syntax."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT title, description, published_at FROM articles WHERE title LIKE ?", (f"%{keyword}%",))
    rows = cursor.fetchall()
    conn.close()

    print(f"\n--- Search Results for '{keyword}' ---")
    if not rows:
        print("No matching articles found.")
        return

    for idx, (title, description, published_at) in enumerate(rows, 1):
        print(f"{idx}. Title: {title}")
        print(f"   Description: {description}")
        print(f"   Published At: {published_at}\n")

def delete_headlines_by_date_range(start_date, end_date):
    """Deletes headlines published within a specific date range (YYYY-MM-DD)."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM articles WHERE published_at BETWEEN ? AND ?", (start_date, end_date))
    deleted_count = cursor.rowcount
    conn.commit()
    conn.close()
    print(f"\nDeleted {deleted_count} articles published between {start_date} and {end_date}.")

# --- News Fetching logic ---
def fetch_news(topic, api_key):
    """Fetches top 5 articles related to the topic from NewsAPI."""
    url = f"https://newsapi.org/v2/everything?q={topic}&pageSize=5&apiKey={api_key}"
    try:
        response = get(url)
        if response.status_code == 200:
            data = response.json()
            return data.get("articles", [])
        else:
            print(f"Error fetching data: Status Code {response.status_code}")
            return []
    except Exception as e:
        print(f"An error occurred during API fetch: {e}")
        return []

# --- Execution flow ---
try:
    api_key = userdata.get("NEWS_API_KEY")
except Exception:
    api_key = input("Enter NewsAPI Key: ").strip()

if api_key:
    init_db()
    topic = input("Enter news topic: ").strip() or "technology"

    print(f"\nRetrieving top 5 headlines for: {topic}...")
    articles = fetch_news(topic, api_key)

    if articles:
        save_headlines(articles)
        display_all_articles()

        # Challenge Task A: Search
        search_choice = input("Would you like to search the database by keyword? (yes/no): ").strip().lower()
        if search_choice == "yes":
            keyword = input("Enter keyword to search: ").strip()
            search_headlines_by_keyword(keyword)

        # Challenge Task B: Delete by Date Range
        delete_choice = input("Would you like to delete old headlines by a date range? (yes/no): ").strip().lower()
        if delete_choice == "yes":
            print("Enter date range in YYYY-MM-DD format:")
            start_date = input("Start Date: ").strip()
            end_date = input("End Date: ").strip()
            delete_headlines_by_date_range(start_date, end_date)
            display_all_articles()
    else:
        print("No articles found or API error occurred.")
else:
    print("API Key is required to proceed.")